In [10]:
import codecs
import numpy as np
import gensim
import textwrap
import torch


In [11]:
torch.cuda.empty_cache()

In [12]:
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

chunk_size = 100

with open(
    "/home/julia/Desktop/code/PAC/RNN/измененные андроиды.txt",
    "r",
    encoding="utf-8",
) as f:
    text = f.read()
    text = text[: len(text) // chunk_size * chunk_size]
    split_text = textwrap.wrap(text, width=chunk_size, break_long_words=False)


all_chars = np.array(list(text)).reshape(-1, 1)
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
encoder.fit(all_chars)
encoding_size = len(encoder.categories_[0])
encoded_all = encoder.transform(all_chars)

n_sentences = encoded_all.shape[0] // chunk_size
res = encoded_all.reshape(n_sentences, chunk_size, encoding_size)

In [13]:
print(res[0])

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [14]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("all cool")
else:
    print("плак плак")   

all cool


китайский разум говорит что данные для обучения устроены так:
х = предложение исходное
у = то же предложение но сдвинутое на один символ вправо

In [15]:
# X: берем все чанки, все символы КРОМЕ последнего, все признаки
X = res[:, :-1, :]

# Y: берем все чанки, все символы КРОМЕ первого, все признаки
Y = res[:, 1:, :]

print(f"Форма X (Вход): {X.shape}")

print(f"Форма Y (Цель): {Y.shape}")

Форма X (Вход): (3923, 99, 92)
Форма Y (Цель): (3923, 99, 92)


In [16]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Y нужно превратить из One-Hot в индексы для CrossEntropyLoss
# Было (4013, 99, 92)  Стало (4013, 99)
Y = torch.from_numpy(Y).float()
X = torch.from_numpy(X).float()
Y_indices = torch.argmax(Y, dim=2)
X = X.to(device)
Y_indices = Y_indices.to(device)

In [ ]:
batch_size = 64
dataset = TensorDataset(X, Y_indices)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)


class ElSheepModel(nn.Module):
    def __init__(self, input_size=92, hidden_size=300, num_layers=2):
        super().__init__()
        # input_size должен совпадать с размером алфавита
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        # Выходной слой должен выдавать размер алфавита
        self.dense = nn.Linear(hidden_size, input_size)

    def forward(self, x):
        # LSTM возвращает (output_for_all_steps, (hidden_state, cell_state))
        # Мы берем только output, hidden нам не нужен явно
        out, _ = self.lstm(x)
        # Применяем линейный слой к каждому шагу последовательности
        out = self.dense(out)
        return out


input_size = res.shape[2]
model = ElSheepModel(input_size=input_size).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=10
)

epochs = 200

best_loss = 100000
patience_count = 0

for epoch in range(epochs):
    epoch_loss = 0
    num_batches = 0

    for batch_X, batch_Y in loader:
        optimizer.zero_grad()

        output = model(batch_X)

        loss = criterion(output.view(-1, input_size), batch_Y.view(-1))

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        num_batches += 1

    avg_loss = epoch_loss / num_batches
    scheduler.step(avg_loss)
    if avg_loss <= best_loss:
        best_loss = avg_loss
        # я не знаю как это работает есть ли в этом мире логика вообще но когда я вот тут забываю сделать patuence=0 модель лучше обучается
    else:
        if patience_count < 10:
            patience_count+=1
        else:
            print("early stop")
            print(f"Эпоха {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")
            break
    print(f"Эпоха {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

Эпоха 1/200, Loss: 3.2351
Эпоха 2/200, Loss: 3.1681
Эпоха 3/200, Loss: 3.1646
Эпоха 4/200, Loss: 2.9574
Эпоха 5/200, Loss: 2.8189
Эпоха 6/200, Loss: 2.6910
Эпоха 7/200, Loss: 2.5330
Эпоха 8/200, Loss: 2.4022
Эпоха 9/200, Loss: 2.2773
Эпоха 10/200, Loss: 2.1743
Эпоха 11/200, Loss: 2.0702
Эпоха 12/200, Loss: 1.9884
Эпоха 13/200, Loss: 1.9165
Эпоха 14/200, Loss: 1.8489
Эпоха 15/200, Loss: 1.7966
Эпоха 16/200, Loss: 1.7502
Эпоха 17/200, Loss: 1.7156
Эпоха 18/200, Loss: 1.6798
Эпоха 19/200, Loss: 1.6498
Эпоха 20/200, Loss: 1.6197
Эпоха 21/200, Loss: 1.5953
Эпоха 22/200, Loss: 1.5711
Эпоха 23/200, Loss: 1.5510
Эпоха 24/200, Loss: 1.5292
Эпоха 25/200, Loss: 1.5138
Эпоха 26/200, Loss: 1.4941
Эпоха 27/200, Loss: 1.4779
Эпоха 28/200, Loss: 1.4610
Эпоха 29/200, Loss: 1.4448
Эпоха 30/200, Loss: 1.4322
Эпоха 31/200, Loss: 1.4213
Эпоха 32/200, Loss: 1.4105
Эпоха 33/200, Loss: 1.3940
Эпоха 34/200, Loss: 1.3823
Эпоха 35/200, Loss: 1.3705
Эпоха 36/200, Loss: 1.3593
Эпоха 37/200, Loss: 1.3444
Эпоха 38/2

In [21]:
def generate_text(model, seed_text, encoder, length=100, temperature=0.8):
    """Генерация текста моделью"""
    model.eval()

    # Кодирование начального текста
    chars = list(seed_text.ljust(100))  # дополняем до 100 символов
    input_chars = np.array(chars).reshape(-1, 1)
    input_encoded = encoder.transform(input_chars)
    input_tensor = (
        torch.from_numpy(input_encoded).float().unsqueeze(0).to(device)
    )  # (1, 100, 92)

    generated = list(seed_text)

    with torch.no_grad():
        for _ in range(length):
            output = model(input_tensor[:, -100:, :])  # последние 100 символов
            next_char_logits = output[:, -1, :]  # логиты для следующего символа

            # Temperature sampling для разнообразия
            if temperature > 0:
                probs = F.softmax(next_char_logits / temperature, dim=1)
                next_idx = torch.multinomial(probs, 1).item()
            else:
                next_idx = torch.argmax(next_char_logits, dim=1).item()

            next_char = encoder.categories_[0][next_idx]
            generated.append(next_char)

            # Добавляем новый символ ко входу
            next_onehot = (
                F.one_hot(torch.tensor(next_idx), num_classes=input_size)
                .float()
                .to(device)
            )
            input_tensor = torch.cat(
                [input_tensor, next_onehot.unsqueeze(0).unsqueeze(0)], dim=1
            )

    return "".join(generated)

for temp in [0.4, 0.7, 1.0]:
    print(f"\n--- Temperature: {temp} ---")
    print(generate_text(model, "Андроиды ", encoder, length=100, temperature=temp))

for temp in [0.4, 0.7, 1.0]:
    print(f"\n--- Temperature: {temp} ---")
    print(generate_text(model, "эмпатия", encoder, length=100, temperature=temp))


--- Temperature: 0.4 ---
Андроиды                Рой Бати   спросила Рейчел                                                           

--- Temperature: 0.7 ---
Андроиды         Давно  унавливая   если  ответила Айрен  Она увелить свою легоным создание и все что вы долж

--- Temperature: 1.0 ---
Андроиды   Новейти взяв  же обновление  набуры  уже
бастерым   Ее жене  шенный често  надпись  Ты  почти
годы

--- Temperature: 0.4 ---
эмпатия                                 По Думаю ее дело в сейчас за Риком  как не отказать ваша белочка  с

--- Temperature: 0.7 ---
эмпатия         Я не андроид  ответил Рик  В коза не сказавших в
коридор и обнаружив на кути  сообщила Айре

--- Temperature: 1.0 ---
эмпатия  Она
Хорожной бумаги Всплидел А
нишей костям уже знаешь и спустись  меня черт во время
андроида  оз


--- Temperature: 0.3 ---
Андроиды добавил  Вы не могли бы подать от своих серуениями заказаниям ВойтКампфа  сказал Рик  В отношении ко

--- Temperature: 0.7 ---
Андроиды она уберение соседнее к этого визор изменения  ответила она  просто от себя руками униздадившись  В 

--- Temperature: 1.0 ---
Андроиды она накотру  сказал Рик
      В данная кабинета  Если только учил из квартире к моные мозго обманнов

--- Temperature: 0.3 ---
эмпатиявашего проверку все что вы тоже и вновь подозрительной память  спросил он самое главное  Она поднялс

--- Temperature: 0.7 ---
эмпатияне делала на собеседный экспелась
      Вот это потянул ей работы изза мной и внимательно обменица к

--- Temperature: 1.0 ---
эмпатиячашкл одну настолько чувственную крузной  насщештя  Он сбраском Изидор вспомнив
долларов а разберей